# Integración de agricultura, clima y edafología (2003–2025)

Esta notebook construye de forma reproducible el conjunto municipal para análisis y modelado. Utiliza los archivos agrícolas oficiales normalizados por el paquete, clima mensual de NASA POWER y polígonos edafológicos de INEGI.

La unidad agrícola original se conserva: la integración no debe crear ni eliminar registros.

In [1]:
import geopandas as gpd
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIOS,
    EDAFOLOGIA_RAW_DIR,
    INEGI_INTERIM_DIR,
    NASA_POWER_PROCESSED_DIR,
    PROCESSED_DATA_DIR,
)
from proyecto_agronomia.features import construir_historico_agricola #carga y normaliza los datos de agricultura
from proyecto_agronomia.processing.integracion_climatica import (
    construir_clima_mensual_ancho,
    crear_clave_municipal,
    integrar_agricultura_clima,
    resumir_clima_por_ciclo,
    validar_clima_mensual,
)
from proyecto_agronomia.processing.integracion_edafologica import (
    resumir_edafologia_municipal,
)
from proyecto_agronomia.processing.features_historicas import (
    COLUMNAS_HISTORICAS,
    agregar_historial_rendimiento,
)

RUTA_CLIMA_MENSUAL = (
    NASA_POWER_PROCESSED_DIR / "clima_municipal_mensual_2002_2025.csv"
)
RUTA_MUNICIPIOS = INEGI_INTERIM_DIR / "municipios_mexico_inegi.geojson"
RUTA_EDAFOLOGIA = (
    EDAFOLOGIA_RAW_DIR / "serie_iii" / "conjunto_de_datos"
    / "conj_nac_inf_edaf_esc_250k_ser_III_area.shp"
)
RUTA_EDAFOLOGIA_MUNICIPAL = PROCESSED_DATA_DIR / "edafologia_municipal.csv"
RUTA_AGRICULTURA_CLIMA = PROCESSED_DATA_DIR / "agricultura_clima_2003_2025.csv"
RUTA_FINAL = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_2003_2025.csv"
)
RUTA_FINAL_HISTORIAL = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_historial_2003_2025.csv"
)
RECALCULAR_EDAFOLOGIA_MUNICIPAL = False
pd.set_option("display.max_columns", None)

2026-11-16 18:33:31.409 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## Cargar y normalizar la producción agrícola

Los archivos `agricola_*.csv` se leen desde `data/raw/agricultura`. La normalización resuelve cambios históricos de columnas, nombres de cultivo, precios, separadores de miles y errores numéricos.

In [2]:
df_agricultura = construir_historico_agricola(
    anios=ANIOS,
    directorio=AGRICULTURA_RAW_DIR,
)
df_agricultura = df_agricultura.drop(
    columns=[c for c in df_agricultura if c.startswith("Unnamed:")],
    errors="ignore",
)
df_agricultura = crear_clave_municipal(df_agricultura)
df_agricultura["Nomcicloproductivo"] = (
    df_agricultura["Nomcicloproductivo"].astype("string").str.strip()
)

columnas_agricolas_requeridas = {
    "Anio", "cvegeo", "Nomestado", "Nommunicipio",
    "Nomcultivo", "Nomcicloproductivo", "Nommodalidad",
    "Sembrada", "Cosechada", "Siniestrada",
    "Volumenproduccion", "Rendimiento",
}
faltantes = columnas_agricolas_requeridas - set(df_agricultura.columns)
assert not faltantes, f"Faltan columnas agrícolas: {sorted(faltantes)}"
assert set(df_agricultura["Anio"].unique()) == set(ANIOS)
assert df_agricultura["cvegeo"].str.fullmatch(r"\d{5}").all()

ciclos_validos = {"Otoño-Invierno", "Primavera-Verano", "Perennes"}
ciclos_encontrados = set(df_agricultura["Nomcicloproductivo"].dropna())
assert ciclos_encontrados <= ciclos_validos, (
    f"Ciclos no contemplados: {sorted(ciclos_encontrados-ciclos_validos)}"
)

print(f"Registros agrícolas: {len(df_agricultura):,}")
display(
    df_agricultura.groupby(["Anio", "Nomcicloproductivo"])
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        cultivos=("Nomcultivo", "nunique"),
    )
)

Procesando datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

2026-11-16 18:33:36.161 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:84 - [2003] Columnas normalizadas.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:00<00:17,  1.25it/s]

2026-11-16 18:33:36.613 | INFO     | proyecto_agronomia.features:construir_historico_agricola:180 - [2003] 35,667 registros procesados.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:01<00:17,  1.25it/s]

2026-11-16 18:33:36.956 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:84 - [2004] Columnas normalizadas.


... (83 outputs left out)

2026-11-16 18:33:54.399 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:84 - [2025] Columnas normalizadas.


Procesando datos agrícolas: 100%|██████████| 23/23 [00:19<00:00,  1.21it/s]


2026-11-16 18:33:54.854 | INFO     | proyecto_agronomia.features:construir_historico_agricola:180 - [2025] 35,902 registros procesados.
Registros agrícolas: 825,089


registros  municipios  cultivos
Anio Nomcicloproductivo                                 
2003 Otoño-Invierno           9858        1897       147
     Perennes                10488        2118       128
     Primavera-Verano        15321        2362       159
2004 Otoño-Invierno           9935        1902       155
     Perennes                10838        2130       129
...                            ...         ...       ...
2024 Perennes                11631        2290       132
     Primavera-Verano        16077        2406       162
2025 Otoño-Invierno           9099        1923       146
     Perennes                11763        2292       132
     Primavera-Verano        15040        2393       156

[69 rows x 3 columns]

## Cargar y validar el clima mensual NASA POWER

La llave climática es `cvegeo + anio + mes`. El año 2002 se necesita para formar octubre–diciembre del ciclo Otoño–Invierno de 2003.

In [3]:
if not RUTA_CLIMA_MENSUAL.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_CLIMA_MENSUAL}. Ejecuta descarga_clima_nasa.ipynb."
    )

columnas_clima = [
    "cvegeo", "anio", "mes", "temperatura_media_mes_c",
    "precipitacion_mes_mm", "dias_esperados",
    "dias_temperatura", "dias_precipitacion",
]
df_clima_mensual = pd.read_csv(
    RUTA_CLIMA_MENSUAL,
    usecols=columnas_clima,
    dtype={"cvegeo": "string"},
)
df_clima_mensual = validar_clima_mensual(df_clima_mensual)
print(f"Registros climáticos mensuales: {len(df_clima_mensual):,}")
display(
    df_clima_mensual.groupby("anio").agg(
        municipios=("cvegeo", "nunique"),
        meses=("mes", "count"),
        temperatura_nulos=("temperatura_media_mes_c", lambda s: s.isna().sum()),
        precipitacion_nulos=("precipitacion_mes_mm", lambda s: s.isna().sum()),
    )
)

Registros climáticos mensuales: 713,664


,municipios,meses,temperatura_nulos,precipitacion_nulos
anio,,,,
2002,2478,29736,0,0
2003,2478,29736,0,0
2004,2478,29736,0,0
2005,2478,29736,0,0
2006,2478,29736,0,0
2007,2478,29736,0,0
2008,2478,29736,0,0
2009,2478,29736,0,0
2010,2478,29736,0,0


## Construir características climáticas

Se conservan los meses como columnas y se calculan ventanas compatibles con los ciclos productivos:

- Otoño–Invierno: octubre–diciembre anterior y enero–marzo actual.
- Primavera–Verano: abril–septiembre.
- Perennes: enero–diciembre.

In [4]:
df_clima_ancho = construir_clima_mensual_ancho(df_clima_mensual)
df_clima_ciclos = resumir_clima_por_ciclo(df_clima_mensual)
df_clima_ciclos = df_clima_ciclos.loc[df_clima_ciclos["anio"].isin(ANIOS)]

assert not df_clima_ancho.duplicated(["cvegeo", "anio"]).any()
assert not df_clima_ciclos.duplicated(
    ["cvegeo", "anio", "Nomcicloproductivo"]
).any()
assert df_clima_ciclos[[
    "temperatura_media_ciclo_c", "precipitacion_ciclo_mm"
]].notna().all().all()

display(
    df_clima_ciclos.groupby(["anio", "Nomcicloproductivo"])
    .agg(
        municipios=("cvegeo", "nunique"),
        dias_min=("dias_clima_ciclo", "min"),
        dias_max=("dias_clima_ciclo", "max"),
    )
)

municipios  dias_min  dias_max
anio Nomcicloproductivo                                
2003 Otoño-Invierno            2478       182       182
     Perennes                  2478       365       365
     Primavera-Verano          2478       183       183
2004 Otoño-Invierno            2478       183       183
     Perennes                  2478       366       366
...                             ...       ...       ...
2024 Perennes                  2478       366       366
     Primavera-Verano          2478       183       183
2025 Otoño-Invierno            2478       182       182
     Perennes                  2478       365       365
     Primavera-Verano          2478       183       183

[69 rows x 3 columns]

## Integrar agricultura y clima

Las uniones son `many_to_one`; por lo tanto, el número de registros agrícolas debe permanecer sin cambios y toda observación debe recibir clima.

In [5]:
df_agricultura_clima = integrar_agricultura_clima(
    df_agricultura,
    df_clima_ciclos,
    df_clima_ancho,
)
columnas_climaticas_finales = [
    columna for columna in df_agricultura_clima.columns
    if columna.startswith(("temperatura_", "precipitacion_"))
]
assert len(df_agricultura_clima) == len(df_agricultura)
assert df_agricultura_clima[columnas_climaticas_finales].notna().all().all()

resumen_cobertura = (
    df_agricultura_clima.groupby(["Anio", "Nomcicloproductivo"])
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        temperatura=("temperatura_media_ciclo_c", "count"),
        precipitacion=("precipitacion_ciclo_mm", "count"),
    )
)
resumen_cobertura["cobertura_pct"] = (
    100 * resumen_cobertura["temperatura"] / resumen_cobertura["registros"]
).round(2)
assert resumen_cobertura["cobertura_pct"].eq(100).all()
display(resumen_cobertura)

RUTA_AGRICULTURA_CLIMA.parent.mkdir(parents=True, exist_ok=True)
df_agricultura_clima.to_csv(
    RUTA_AGRICULTURA_CLIMA, index=False, encoding="utf-8"
)
print(f"Dataset agricultura-clima: {RUTA_AGRICULTURA_CLIMA}")
print(f"Dimensiones: {df_agricultura_clima.shape}")

registros  municipios  temperatura  precipitacion  \
Anio Nomcicloproductivo                                                      
2003 Otoño-Invierno           9858        1897         9858           9858   
     Perennes                10488        2118        10488          10488   
     Primavera-Verano        15321        2362        15321          15321   
2004 Otoño-Invierno           9935        1902         9935           9935   
     Perennes                10838        2130        10838          10838   
...                            ...         ...          ...            ...   
2024 Perennes                11631        2290        11631          11631   
     Primavera-Verano        16077        2406        16077          16077   
2025 Otoño-Invierno           9099        1923         9099           9099   
     Perennes                11763        2292        11763          11763   
     Primavera-Verano        15040        2393        15040          15040   

                         cobertura_pct  
Anio Nomcicloproductivo                 
2003 Otoño-Invierno              100.0  
     Perennes                    100.0  
     Primavera-Verano            100.0  
2004 Otoño-Invierno              100.0  
     Perennes                    100.0  
...                                ...  
2024 Perennes                    100.0  
     Primavera-Verano            100.0  
2025 Otoño-Invierno              100.0  
     Perennes                    100.0  
     Primavera-Verano            100.0  

[69 rows x 5 columns]

Dataset agricultura-clima: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_2003_2025.csv
Dimensiones: (825089, 58)


## Construir o cargar el resumen edafológico municipal

La intersección espacial puede tardar. Una vez generada, `edafologia_municipal.csv` se reutiliza. Cambia `RECALCULAR_EDAFOLOGIA_MUNICIPAL` a `True` solamente para reconstruirla.

In [6]:
if RUTA_EDAFOLOGIA_MUNICIPAL.exists() and not RECALCULAR_EDAFOLOGIA_MUNICIPAL:
    df_edafologia_municipal = pd.read_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, dtype={"cvegeo": "string"}
    )
    df_edafologia_municipal["cvegeo"] = (
        df_edafologia_municipal["cvegeo"].str.zfill(5)
    )
else:
    for ruta in [RUTA_MUNICIPIOS, RUTA_EDAFOLOGIA]:
        if not ruta.exists():
            raise FileNotFoundError(f"No se encontró el archivo requerido: {ruta}")
    municipios = gpd.read_file(RUTA_MUNICIPIOS)
    edafologia = gpd.read_file(RUTA_EDAFOLOGIA)
    df_edafologia_municipal = resumir_edafologia_municipal(
        municipios, edafologia
    )
    df_edafologia_municipal.to_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, index=False, encoding="utf-8"
    )

assert not df_edafologia_municipal["cvegeo"].duplicated().any()
print(f"Municipios edafológicos: {len(df_edafologia_municipal):,}")
display(df_edafologia_municipal.head())

Municipios edafológicos: 2,478


,cvegeo,grupo_suelo_dominante,grupo_suelo_dominante_pct,textura_dominante,textura_dominante_pct,fase_fisica_dominante,fase_fisica_dominante_pct,fase_quimica_dominante,fase_quimica_dominante_pct,grupos_suelo_n,nombre_suelo_wrb
0,01001,PH,24.72,2.0,88.81,lep,35.30,NaN,NaN,11,Phaeozem
1,01008,LP,31.44,2.0,95.48,lep,47.28,so,0.49,10,Leptosol
2,01011,DU,41.24,2.0,93.67,pdn,33.27,NaN,NaN,7,Durisol
3,01009,DU,38.81,2.0,77.62,pdn,27.35,NaN,NaN,6,Durisol
4,01007,DU,39.54,2.0,98.41,lep,51.24,so,9.54,11,Durisol


## Integrar suelo y guardar el dataset final

El resumen edafológico contiene una fila por municipio, por lo que la unión tampoco debe multiplicar registros agrícolas.

In [7]:
df_final = df_agricultura_clima.merge(
    df_edafologia_municipal,
    on="cvegeo",
    how="left",
    validate="many_to_one",
)
assert len(df_final) == len(df_agricultura), (
    "La integración edafológica alteró el número de registros."
)
columnas_suelo = [
    "grupo_suelo_dominante", "nombre_suelo_wrb",
    "grupo_suelo_dominante_pct", "grupos_suelo_n",
    "textura_dominante",
]
faltantes_suelo = df_final["grupo_suelo_dominante"].isna()
if faltantes_suelo.any():
    claves_faltantes = sorted(df_final.loc[faltantes_suelo, "cvegeo"].unique())
    raise ValueError(
        f"Hay {len(claves_faltantes)} municipios agrícolas sin suelo: "
        f"{claves_faltantes[:20]}"
    )

df_final.to_csv(RUTA_FINAL, index=False, encoding="utf-8")
print(f"Dataset final: {RUTA_FINAL}")
print(f"Registros: {len(df_final):,}")
print(f"Columnas: {len(df_final.columns):,}")
print(f"Periodo: {df_final['Anio'].min()}–{df_final['Anio'].max()}")
display(
    df_final[[
        "Anio", "cvegeo", "Nomestado", "Nommunicipio",
        "Nomcultivo", "Nomcicloproductivo",
        "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
        *columnas_suelo,
    ]].head()
)

Dataset final: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_2003_2025.csv
Registros: 825,089
Columnas: 68
Periodo: 2003–2025


,Anio,cvegeo,Nomestado,Nommunicipio,Nomcultivo,Nomcicloproductivo,temperatura_media_ciclo_c,precipitacion_ciclo_mm,grupo_suelo_dominante,nombre_suelo_wrb,grupo_suelo_dominante_pct,grupos_suelo_n,textura_dominante
0,2003,01001,Aguascalientes,Aguascalientes,Avena forrajera en verde,Otoño-Invierno,13.835165,84.8,PH,Phaeozem,24.72,11,2.0
1,2003,01001,Aguascalientes,Aguascalientes,Cebada forrajera en verde,Otoño-Invierno,13.835165,84.8,PH,Phaeozem,24.72,11,2.0
2,2003,01001,Aguascalientes,Aguascalientes,Centeno forrajero en verde,Otoño-Invierno,13.835165,84.8,PH,Phaeozem,24.72,11,2.0
3,2003,01001,Aguascalientes,Aguascalientes,Triticale forrajero en verde,Otoño-Invierno,13.835165,84.8,PH,Phaeozem,24.72,11,2.0
4,2003,01001,Aguascalientes,Aguascalientes,Pastos y praderas,Otoño-Invierno,13.835165,84.8,PH,Phaeozem,24.72,11,2.0


## Agregar estabilidad histórica sin fuga temporal

Para cada municipio, cultivo, ciclo, modalidad y unidad, las características del año actual utilizan exclusivamente rendimientos de años anteriores. Los primeros años permanecen sin historial suficiente.

In [8]:
df_final_historial = agregar_historial_rendimiento(df_final, minimo_periodos=3)
assert len(df_final_historial) == len(df_final)
assert df_final_historial.loc[
    df_final_historial['Anio'].eq(df_final_historial['Anio'].min()),
    'rendimiento_hist_media',
].isna().all()
df_final_historial.to_csv(RUTA_FINAL_HISTORIAL, index=False, encoding='utf-8')
print(f'Dataset con historial: {RUTA_FINAL_HISTORIAL}')
display(
    df_final_historial[[
        'Anio', 'cvegeo', 'Nomcultivo', 'Nomcicloproductivo',
        'Nommodalidad', *COLUMNAS_HISTORICAS,
    ]].loc[lambda tabla: tabla['rendimiento_hist_media'].notna()].head()
)

Dataset con historial: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_historial_2003_2025.csv


,Anio,cvegeo,Nomcultivo,Nomcicloproductivo,Nommodalidad,rendimiento_hist_anios,rendimiento_anio_anterior,rendimiento_hist_media,rendimiento_hist_desviacion,rendimiento_hist_cv
108391,2006,01001,Avena forrajera en verde,Otoño-Invierno,Riego,3,26.370840,26.702666,0.287415,0.010764
108393,2006,01001,Triticale forrajero en verde,Otoño-Invierno,Riego,3,25.333333,25.111111,5.003702,0.199262
108394,2006,01001,Pastos y praderas,Otoño-Invierno,Riego,3,59.918715,60.947866,1.183411,0.019417
108396,2006,01001,Chile verde,Primavera-Verano,Riego,3,6.485714,6.281501,0.348771,0.055524
108397,2006,01001,Coliflor,Primavera-Verano,Riego,3,12.000000,11.291667,1.120361,0.099220


## Recargar y validar el producto

Esta última lectura verifica que el CSV persistido conserva la llave municipal, el periodo y las variables esenciales.

In [9]:
df_verificacion = pd.read_csv(
    RUTA_FINAL_HISTORIAL, dtype={"cvegeo": "string"}, low_memory=False
)
df_verificacion["cvegeo"] = df_verificacion["cvegeo"].str.zfill(5)
assert len(df_verificacion) == len(df_final)
assert set(df_verificacion["Anio"].unique()) == set(ANIOS)
assert df_verificacion["cvegeo"].str.fullmatch(r"\d{5}").all()
assert df_verificacion[[
    "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
    "grupo_suelo_dominante",
]].notna().all().all()
assert set(COLUMNAS_HISTORICAS).issubset(df_verificacion.columns)
display(pd.DataFrame({
    "archivo": [str(RUTA_FINAL_HISTORIAL)],
    "registros": [len(df_verificacion)],
    "columnas": [len(df_verificacion.columns)],
    "municipios": [df_verificacion["cvegeo"].nunique()],
    "cultivos": [df_verificacion["Nomcultivo"].nunique()],
    "anio_inicial": [df_verificacion["Anio"].min()],
    "anio_final": [df_verificacion["Anio"].max()],
}))

,archivo,registros,columnas,municipios,cultivos,anio_inicial,anio_final
0,B:\MCD2\JW\agro\proyecto_agronomia\data\proces...,825089,73,2452,385,2003,2025
